<a href="https://colab.research.google.com/github/hquinnett/final-project-GB885-quinnett-h/blob/main/GB885_Final_Project_Draft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RUSH Sales Analysis

Hailey Quinnett  
GB 885 - Python Fundamentals  
Final Project  
8/9/2026

## Overview

RUSH is a globally renowned sportswear and footwear brand known for its innovative designs and performance-oriented products. This project analyzes RUSH's raw US sales data from 2020 and 2021 to identify trends and insights that can help company leadership understand the market and identify opportunities for growth.

The company stores its raw sales data as a collection of three tables:

*   TABLE_PRODUCTS
*   TABLE_RETAILER
*   TABLE_SALES

The data includes the number of units sold, the total sales revenue, the location of the sales, the type of product sold, as well as other relevant information.

## Business Questions

The VP of US Sales asked for answers to the following questions:

1.   What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?
2.   What state had the highest sales (in dollars) of women's products in 2021? How much was it?
3.   What state had the highest sales (in dollars) of men's products in 2021? How much was it?
4.   What retailer purchased the most units in 2021? In 2020?





## Import libraries & datasets

In [79]:
# Import libraries
import pandas as pd
import sklearn

In [13]:
# Load data from github repository
df_sales = pd.read_csv('https://raw.githubusercontent.com/hquinnett/final-project-GB885-quinnett-h/refs/heads/main/data/TABLE_SALES_885.csv')
df_retailer = pd.read_csv('https://raw.githubusercontent.com/hquinnett/final-project-GB885-quinnett-h/refs/heads/main/data/TABLE_RETAILER_885.csv')
# Set delimiter to '|' for products csv
df_products = pd.read_csv('https://raw.githubusercontent.com/hquinnett/final-project-GB885-quinnett-h/refs/heads/main/data/TABLE_PRODUCTS_885.csv', delimiter='|')

In [14]:
# Preview sales data
df_sales.head()

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD
0,1,A00MOHCO,1/1/2020,1,1,2020,20,50.0,1200,0.5,In-store
1,7,A00MOHCO,1/7/2020,1,7,2020,20,50.0,1250,0.5,In-store
2,13,A00MOHCO,1/25/2020,1,25,2020,20,50.0,1220,0.5,Outlet
3,19,A00MOHCO,1/31/2020,1,31,2020,20,50.0,1200,0.5,Outlet
4,25,A00MOHCO,2/6/2020,2,6,2020,20,60.0,1220,0.5,Outlet


In [15]:
# Preview retailer data
df_retailer.head()

,RETAILER_ID,RETAILER,REGION,STATE,CITY
0,A00MOHCO,Amazon,Midwest,Ohio,Columbus
1,A00NMAPO,Amazon,Northeast,Maine,Portland
2,A00NMABO,Amazon,Northeast,Massachusetts,Boston
3,A00NNEMA,Amazon,Northeast,New Hampshire,Manchester
4,A00NVEBU,Amazon,Northeast,Vermont,Burlington


In [16]:
# Preview products data
df_products.head()

,PRODUCT_ID,PRODUCT_NAME
0,20,Men's Street Footwear
1,30,Men's Athletic Footwear
2,120,Women's Street Footwear
3,130,Women's Athletic Footwear
4,40,Men's Apparel


In [17]:
# Merge retailer and sales dataframes
df = df_sales.merge(df_retailer, on='RETAILER_ID', how='left')

# Merge product dataframe into the main dataframe
df= df.merge(df_products, on= 'PRODUCT_ID', how='left')

# Preview merged data
df.head()

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD,RETAILER,REGION,STATE,CITY,PRODUCT_NAME
0,1,A00MOHCO,1/1/2020,1,1,2020,20,50.0,1200,0.5,In-store,Amazon,Midwest,Ohio,Columbus,Men's Street Footwear
1,7,A00MOHCO,1/7/2020,1,7,2020,20,50.0,1250,0.5,In-store,Amazon,Midwest,Ohio,Columbus,Men's Street Footwear
2,13,A00MOHCO,1/25/2020,1,25,2020,20,50.0,1220,0.5,Outlet,Amazon,Midwest,Ohio,Columbus,Men's Street Footwear
3,19,A00MOHCO,1/31/2020,1,31,2020,20,50.0,1200,0.5,Outlet,Amazon,Midwest,Ohio,Columbus,Men's Street Footwear
4,25,A00MOHCO,2/6/2020,2,6,2020,20,60.0,1220,0.5,Outlet,Amazon,Midwest,Ohio,Columbus,Men's Street Footwear


## Data inspection

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10271 entries, 0 to 10270
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORDER_ID          10271 non-null  int64  
 1   RETAILER_ID       10271 non-null  object 
 2   INVOICE_DATE      10271 non-null  object 
 3   MONTH             10271 non-null  int64  
 4   DAY               10271 non-null  int64  
 5   YEAR              10271 non-null  int64  
 6   PRODUCT_ID        10271 non-null  int64  
 7   PRICE_PER_UNIT    10269 non-null  float64
 8   UNITS_SOLD        10271 non-null  object 
 9   OPERATING_MARGIN  10271 non-null  float64
 10  SALES_METHOD      10271 non-null  object 
 11  RETAILER          10270 non-null  object 
 12  REGION            10270 non-null  object 
 13  STATE             10270 non-null  object 
 14  CITY              10270 non-null  object 
 15  PRODUCT_NAME      10271 non-null  object 
dtypes: float64(2), int64(5), object(9)
memor

In [19]:
# Features not needed for the analysis: ORDER_ID, RETAILER_ID, PRODUCT_ID
df = df.drop(['ORDER_ID', 'RETAILER_ID', 'PRODUCT_ID'], axis=1)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10271 entries, 0 to 10270
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   INVOICE_DATE      10271 non-null  object 
 1   MONTH             10271 non-null  int64  
 2   DAY               10271 non-null  int64  
 3   YEAR              10271 non-null  int64  
 4   PRICE_PER_UNIT    10269 non-null  float64
 5   UNITS_SOLD        10271 non-null  object 
 6   OPERATING_MARGIN  10271 non-null  float64
 7   SALES_METHOD      10271 non-null  object 
 8   RETAILER          10270 non-null  object 
 9   REGION            10270 non-null  object 
 10  STATE             10270 non-null  object 
 11  CITY              10270 non-null  object 
 12  PRODUCT_NAME      10271 non-null  object 
dtypes: float64(2), int64(3), object(8)
memory usage: 1.0+ MB


In [21]:
# UNITS_SOLD has data type object - check for nonint values

# Inspect unique values
print(df['UNITS_SOLD'].unique())

['1200' '1250' '1220' '1275' '1150' '900' '850' '875' '925' '950' '800'
 '775' '825' '975' '1025' '700' '750' '475' '525' '495' '450' '500' '575'
 '550' '425' '400' '725' '675' '625' '445' '470' '375' '600' '1050' '1100'
 '1070' '1075' '1125' '1000' '350' '650' '420' '325' '300' '545' '570'
 '870' '820' '770' '795' '620' '670' '1020' '1045' '920' '945' '520' '720'
 '745' '645' '595' '336' '313' '354' '317' '319' '360' '299' '312' '234'
 '213' '254' '268' '238' '255' '240' '209' '223' '230' '228' '252' '273'
 '277' '241' '236' '263' '293' '203' '225' '216' '208' '270' '224' '231'
 '124' '158' '144' '122' '134' '135' '161' '149' '115' '112' '150' '189'
 '176' '182' '195' '181' '156' '111' '119' '120' '136' '145' '143' '104'
 '113' '160' '218' '210' '174' '138' '155' '196' '123' '131' '165' '202'
 '162' '137' '117' '167' '239' '217' '188' '256' '290' '304' '284' '244'
 '323' '116' '106' '98' '91' '173' '169' '180' '215' '233' '200' '105'
 '118' '94' '84' '163' '206' '175' '140' '127' '147

In [22]:
# Check rows containing '***' for UNITS_SOLD
df[df['UNITS_SOLD'] == '***']

,INVOICE_DATE,MONTH,DAY,YEAR,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD,RETAILER,REGION,STATE,CITY,PRODUCT_NAME
1021,5/27/2021,5,27,2021,51.0,***,0.45,Online,Sports Direct,South,Alabama,Birmingham,Men's Street Footwear
1527,12/10/2021,12,10,2021,29.0,***,0.46,Outlet,West Gear,Midwest,Iowa,Des Moines,Men's Street Footwear


In [25]:
# Change '***' for UNITS_SOLD to null
df['UNITS_SOLD'] = df['UNITS_SOLD'].replace('***', pd.NA)

# check
df[df['UNITS_SOLD'] == '***']

,INVOICE_DATE,MONTH,DAY,YEAR,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD,RETAILER,REGION,STATE,CITY,PRODUCT_NAME


In [29]:
# Convert UNITS_SOLD column to numeric
df['UNITS_SOLD'] = pd.to_numeric(df['UNITS_SOLD'])

# Check
df['UNITS_SOLD'].dtype

dtype('float64')

In [28]:
# INVOICE_DATE has data type object

# Inspect unique values
print(df['INVOICE_DATE'].unique())

['1/1/2020' '1/7/2020' '1/25/2020' '1/31/2020' '2/6/2020' '3/4/2020'
 '3/10/2020' '3/16/2020' '4/19/2020' '4/27/2020' '5/3/2020' '7/19/2020'
 '7/25/2020' '7/31/2020' '8/6/2020' '8/12/2020' '8/18/2020' '8/24/2020'
 '8/30/2020' '9/5/2020' '9/11/2020' '10/20/2020' '10/26/2020' '11/1/2020'
 '11/7/2020' '11/13/2020' '12/25/2020' '12/31/2020' '1/6/2021' '1/12/2021'
 '1/18/2021' '1/24/2021' '1/30/2021' '2/5/2021' '2/11/2021' '2/17/2021'
 '3/13/2021' '3/19/2021' '4/7/2021' '4/13/2021' '4/19/2021' '4/25/2021'
 '5/1/2021' '5/7/2021' '5/13/2021' '5/19/2021' '5/25/2021' '5/31/2021'
 '6/6/2021' '6/12/2021' '6/18/2021' '6/24/2021' '6/30/2021' '7/6/2021'
 '7/12/2021' '7/18/2021' '7/24/2021' '7/30/2021' '8/5/2021' '8/11/2021'
 '8/17/2021' '8/23/2021' '8/29/2021' '9/4/2021' '9/10/2021' '9/16/2021'
 '9/22/2021' '9/28/2021' '10/4/2021' '10/10/2021' '10/16/2021'
 '10/22/2021' '10/28/2021' '11/3/2021' '11/9/2021' '11/15/2021'
 '11/21/2021' '11/27/2021' '12/3/2021' '12/9/2021' '12/15/2021'
 '12/21/2021' '12

In [30]:
# Convert INVOICE_DATE to datetime
df['INVOICE_DATE'] = pd.to_datetime(df['INVOICE_DATE'])

# Check
df['INVOICE_DATE'].dtype

dtype('<M8[ns]')

In [32]:
# Inspect for traditional null values
df.isnull().sum()

,0
INVOICE_DATE,0
MONTH,0
DAY,0
YEAR,0
PRICE_PER_UNIT,2
UNITS_SOLD,2
OPERATING_MARGIN,0
SALES_METHOD,0
RETAILER,1
REGION,1


In [34]:
# Inspect for non traditional missing values - categorical

# list of categorical variables
cat_var = list(df.select_dtypes(include=['object']).columns)

# view unique values for each of those variables
for column in cat_var:
  print(column)
  print(df[column].unique())

SALES_METHOD
['In-store' 'Outlet' 'Ootlet' 'Online']
RETAILER
['Amazon' 'Foot Locker' "Kohl's" 'Sports Direct' 'Walmart' 'West Gear' nan]
REGION
['Midwest' 'Northeast' 'South' 'Southeast' 'West' nan]
STATE
['Ohio' 'Maine' 'Massachusetts' 'New Hampshire' 'Vermont' 'Alabama'
 'Kentucky' 'North Carolina' 'Alaska' 'Louisiana' 'Georgia' 'Virginia'
 'Arizona' 'Hawaii' 'Nebraska' 'Idaho' 'Wyoming' 'Illinois' 'Iowa'
 'Kansas' 'Michigan' 'Minnesota' 'Missouri' 'North Dakota' 'South Dakota'
 'Connecticut' 'Delaware' 'Maryland' 'Florida' 'New York' 'Pennsylvania'
 'Rhode Island' 'South Carolina' 'West Virginia' 'Mississippi' 'Tennessee'
 'Texas' 'California' 'Washington' 'New Mexico' 'New Jersey' 'Montana'
 'Oklahoma' 'Arkansas' 'Colorado' 'Nevada' 'Oregon' 'Utah' 'Indiana'
 'Wisconsin' nan]
CITY
['Columbus' 'Portland' 'Boston' 'Manchester' 'Burlington' 'Birmingham'
 'Louisville' 'Charlotte' 'Anchorage' 'New Orleans' 'Atlanta' 'Richmond'
 'Phoenix' 'Honolulu' 'Omaha' 'Boise' 'Cheyenne' 'Chicago' 

In [35]:
# Inspect for non traditional missing values - numerical (99999)
df.describe()

,INVOICE_DATE,MONTH,DAY,YEAR,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN
count,10271,10271.000000,10271.000000,10271.000000,10269.000000,10269.000000,10271.000000
mean,2021-05-13 12:00:54.678220288,6.456333,14.620290,2020.873235,54.575324,246.638232,0.426380
min,2020-01-01 00:00:00,1.000000,1.000000,2020.000000,7.000000,0.000000,0.100000
25%,2021-02-18 00:00:00,3.000000,9.000000,2021.000000,35.000000,100.000000,0.350000
50%,2021-06-05 00:00:00,6.000000,14.000000,2021.000000,45.000000,173.000000,0.420000
75%,2021-09-17 00:00:00,9.000000,20.000000,2021.000000,55.000000,325.000000,0.500000
max,2021-12-31 00:00:00,12.000000,31.000000,2021.000000,99999.000000,1275.000000,0.800000
std,NaN,3.461826,7.301233,0.332725,986.473924,211.881078,0.096018


In [36]:
# Inspect for duplicates
df.duplicated().sum()

np.int64(4)

In [37]:
# List duplicated rows
df[df.duplicated()]

,INVOICE_DATE,MONTH,DAY,YEAR,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD,RETAILER,REGION,STATE,CITY,PRODUCT_NAME
215,2021-01-10,1,10,2021,35.0,375.0,0.25,Outlet,Foot Locker,West,Arizona,Phoenix,Men's Street Footwear
1928,2021-01-10,1,10,2021,45.0,375.0,0.20,Outlet,Foot Locker,West,Arizona,Phoenix,Men's Athletic Footwear
5349,2021-01-10,1,10,2021,45.0,375.0,0.25,Outlet,Foot Locker,West,Arizona,Phoenix,Women's Street Footwear
7061,2021-01-10,1,10,2021,45.0,225.0,0.25,Outlet,Foot Locker,West,Arizona,Phoenix,Women's Athletic Footwear


In [39]:
# Inspect for erroneous values - numerical
df.describe()

,INVOICE_DATE,MONTH,DAY,YEAR,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN
count,10271,10271.000000,10271.000000,10271.000000,10269.000000,10269.000000,10271.000000
mean,2021-05-13 12:00:54.678220288,6.456333,14.620290,2020.873235,54.575324,246.638232,0.426380
min,2020-01-01 00:00:00,1.000000,1.000000,2020.000000,7.000000,0.000000,0.100000
25%,2021-02-18 00:00:00,3.000000,9.000000,2021.000000,35.000000,100.000000,0.350000
50%,2021-06-05 00:00:00,6.000000,14.000000,2021.000000,45.000000,173.000000,0.420000
75%,2021-09-17 00:00:00,9.000000,20.000000,2021.000000,55.000000,325.000000,0.500000
max,2021-12-31 00:00:00,12.000000,31.000000,2021.000000,99999.000000,1275.000000,0.800000
std,NaN,3.461826,7.301233,0.332725,986.473924,211.881078,0.096018


In [41]:
# Inspect for erroneous values - categorical
cat_var = list(df.select_dtypes(include=['object']).columns)

for column in cat_var:
  print(column)
  print(df[[column]].value_counts())

SALES_METHOD
SALES_METHOD
Online          5417
Outlet          3094
In-store        1740
Ootlet            20
Name: count, dtype: int64
RETAILER
RETAILER     
West Gear        2823
Foot Locker      2637
Sports Direct    2088
Kohl's           1030
Amazon            949
Walmart           743
Name: count, dtype: int64
REGION
REGION   
West         2448
Northeast    2432
South        2106
Midwest      1871
Southeast    1413
Name: count, dtype: int64
STATE
STATE         
Texas             594
Florida           549
California        432
Arkansas          432
New York          404
Alabama           216
Louisiana         216
Connecticut       216
Georgia           216
Arizona           216
Mississippi       216
Oklahoma          216
New Mexico        216
New Hampshire     216
Massachusetts     216
Nevada            216
Idaho             216
Rhode Island      216
Oregon            216
Utah              216
Vermont           216
Tennessee         216
Virginia          216
Pennsylvania      216
N

In [43]:
# Use IQR to check for outliers
def count_iqr_outliers(df, column):
    # define q1
    q1 = df[column].quantile(0.25)
    # define q3
    q3 = df[column].quantile(0.75)
    # define iqr
    iqr = q3 - q1
    # define outlier thresholds
    l_threshold = q1 - 1.5 * iqr
    u_threshold = q3 + 1.5 * iqr
    # dount outliers
    outliers = (df[column] < l_threshold) | (df[column] > u_threshold)
    # Count the number of True values (outliers)
    return outliers.sum()

num_var = list(df.select_dtypes(include=['int64', 'float64']).columns)

for column in num_var:
  print(f'{column} : {count_iqr_outliers(df, column)}')

MONTH : 0
DAY : 0
YEAR : 1302
PRICE_PER_UNIT : 85
UNITS_SOLD : 680
OPERATING_MARGIN : 31


### Clean data

In [45]:
# Examine UNITS_SOLD missing values
df[df['UNITS_SOLD'].isnull()]

,INVOICE_DATE,MONTH,DAY,YEAR,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD,RETAILER,REGION,STATE,CITY,PRODUCT_NAME
1021,2021-05-27,5,27,2021,51.0,NaN,0.45,Online,Sports Direct,South,Alabama,Birmingham,Men's Street Footwear
1527,2021-12-10,12,10,2021,29.0,NaN,0.46,Outlet,West Gear,Midwest,Iowa,Des Moines,Men's Street Footwear


In [48]:
# Remove rows with missing units sold
df = df.dropna(subset=['UNITS_SOLD'])
df['UNITS_SOLD'].isnull().sum()

np.int64(0)

In [44]:
# Examine PRICE_PER_UNIT missing values
df[df['PRICE_PER_UNIT'].isnull()]

,INVOICE_DATE,MONTH,DAY,YEAR,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD,RETAILER,REGION,STATE,CITY,PRODUCT_NAME
98,2020-04-02,4,2,2020,NaN,525.0,0.35,In-store,Amazon,Northeast,Vermont,Burlington,Men's Street Footwear
99,2020-04-08,4,8,2020,NaN,525.0,0.50,In-store,Amazon,Northeast,Vermont,Burlington,Men's Street Footwear


In [51]:
# Remove rows with missing price per unit
df = df.dropna(subset=['PRICE_PER_UNIT'])
df['PRICE_PER_UNIT'].isnull().sum()

np.int64(0)

In [54]:
# Examine PRICE_PER_UNIT values of 99999
(df['PRICE_PER_UNIT'] == 99999).sum()

np.int64(1)

In [56]:
# Remove row with 99999 PRICE_PER_UNIT
df = df[df['PRICE_PER_UNIT'] != 99999]
(df['PRICE_PER_UNIT'] == 99999).sum()

np.int64(0)

In [57]:
# Examine rows with missing retailer information
df[
    df['RETAILER'].isnull() |
    df['REGION'].isnull() |
    df['STATE'].isnull() |
    df['CITY'].isnull()
]

,INVOICE_DATE,MONTH,DAY,YEAR,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD,RETAILER,REGION,STATE,CITY,PRODUCT_NAME
1534,2021-07-23,7,23,2021,60.0,298.0,0.42,Outlet,NaN,NaN,NaN,NaN,Men's Street Footwear


In [58]:
# Determine retailer info from sales dataframe
df_sales.loc[1534]

,1534
ORDER_ID,9196
RETAILER_ID,W00WUTSA
INVOICE_DATE,11/20/2021
MONTH,11
DAY,20
YEAR,2021
PRODUCT_ID,20
PRICE_PER_UNIT,18.0
UNITS_SOLD,176
OPERATING_MARGIN,0.46


In [65]:
# PUll missing retailer info from retailer dataframe
df_retailer[df_retailer['RETAILER_ID'] == 'W00WUTSA']

,RETAILER_ID,RETAILER,REGION,STATE,CITY
108,W00WUTSA,West Gear,West,Utah,Salt Lake City


In [66]:
# Add recovered retailer info to main dataframe
df.loc[1534, ['RETAILER', 'REGION', 'STATE', 'CITY']] = ['West Gear', 'West', 'Utah', 'Salt Lake City']

In [69]:
# Drop duplicate Values
df = df.drop_duplicates()

In [71]:
# Replace erroneous value "Ootlet" with "Outlet"
df.replace(to_replace='Ootlet', value = "Outlet", inplace = True)

In [73]:
#re-evaluate outliers post cleaning
#list of numerical variables
num_var = list(df.select_dtypes(include=['int64', 'float64']).columns)

for column in num_var:
    print(column)
    print(count_iqr_outliers(df, column))

MONTH
0
DAY
0
YEAR
1300
PRICE_PER_UNIT
84
UNITS_SOLD
680
OPERATING_MARGIN
31


In [75]:
# Check year values
df['YEAR'].value_counts()

,count
YEAR,
2021,8962
2020,1300


In [78]:
# Windsorize price per unit, units sold, operating margin
wind_var = ['PRICE_PER_UNIT', 'UNITS_SOLD', 'OPERATING_MARGIN']

for column in wind_var:
  #set upper clipping threshold
  high_percentile_value = df[column].quantile(0.95)
  #clip upper outliers
  df.loc[:,column] = df[column].clip(upper=high_percentile_value)
  #set clipping threshold
  low_percentile_value = df[column].quantile(0.05)
  #clip outliers
  df.loc[:,column] = df[column].clip(lower=low_percentile_value)

#check work
for column in wind_var:
    print(column)
    print(count_iqr_outliers(df, column))

PRICE_PER_UNIT
0
UNITS_SOLD
680
OPERATING_MARGIN
0


### Analysis

In [101]:
# Create a total sales field
df['TOTAL_SALES'] = df['PRICE_PER_UNIT'] * df['UNITS_SOLD']

# Separate data by year
df_2020 = df[df['YEAR'] == 2020]
df_2021 = df[df['YEAR'] == 2021]

In [102]:
# Calculate sales for each product category in 2021
df_2021.groupby('PRODUCT_NAME')['TOTAL_SALES'].sum().sort_values()

,TOTAL_SALES
PRODUCT_NAME,
Women's Athletic Footwear,11358774.0
Men's Apparel,13149459.0
Women's Street Footwear,13585183.0
Men's Athletic Footwear,16169860.0
Women's Apparel,18793698.0
Men's Street Footwear,22011662.0


In [105]:
# Calculate sales for women's products for each state in 2021
df_2021[df_2021['PRODUCT_NAME'].str.contains('''Women's''')].groupby('STATE')['TOTAL_SALES'].sum().sort_values(ascending =False)

,TOTAL_SALES
STATE,
Maine,2048624.0
Delaware,1917450.0
New Hampshire,1843175.0
New York,1731765.0
Illinois,1722790.0
Arizona,1706500.0
Virginia,1655690.0
Nebraska,1643305.0
Missouri,1564117.0


In [106]:
# Calculate sales for men's products for each state in 2021
df_2021[df_2021['PRODUCT_NAME'].str.contains('''Men's''')].groupby('STATE')['TOTAL_SALES'].sum().sort_values(ascending =False)

,TOTAL_SALES
STATE,
Arizona,2158900.0
New Hampshire,2133600.0
Delaware,2090050.0
New York,2066506.0
Illinois,1953270.0
Maine,1927915.0
New Mexico,1708626.0
Missouri,1695364.0
Connecticut,1684619.0


In [107]:
# Calculate how many units each retailer purchased in 2021
df_2021.groupby('RETAILER')['UNITS_SOLD'].sum().sort_values()

,UNITS_SOLD
RETAILER,
Walmart,59872.0
Kohl's,137451.0
Amazon,195915.0
Sports Direct,257557.0
West Gear,318103.0
Foot Locker,1069300.0


In [108]:
# Calculate how many units each retailer purchased in 2020
df_2020.groupby('RETAILER')['UNITS_SOLD'].sum().sort_values()

,UNITS_SOLD
RETAILER,
Sports Direct,18822.0
West Gear,57334.0
Kohl's,68694.0
Amazon,278625.0


In [110]:
# Calculate total sales for each retailer in 2021
df_2021.groupby('RETAILER')['TOTAL_SALES'].sum().sort_values()

,TOTAL_SALES
RETAILER,
Walmart,2329931.0
Kohl's,7229133.0
Amazon,9914000.0
Sports Direct,11966621.0
West Gear,12578751.0
Foot Locker,51050200.0


In [111]:
# Calculate total sales for each retailer in 2020
df_2020.groupby('RETAILER')['TOTAL_SALES'].sum().sort_values()

,TOTAL_SALES
RETAILER,
Sports Direct,916653.0
West Gear,2179209.0
Kohl's,3567325.0
Amazon,15117125.0


In [112]:
# Calculate sales for each sales method in 2021
df_2021.groupby('SALES_METHOD')['TOTAL_SALES'].sum().sort_values()

,TOTAL_SALES
SALES_METHOD,
In-store,25279150.0
Outlet,28562180.0
Online,41227306.0


In [113]:
# Calculate sales for each sales method in 2020
df_2020.groupby('SALES_METHOD')['TOTAL_SALES'].sum().sort_values()

,TOTAL_SALES
SALES_METHOD,
Online,4483978.0
In-store,8340325.0
Outlet,8956009.0


In [115]:
# Calculate sales in 2021 broken down by retailer and sales medthod
df_2021.groupby(['RETAILER', 'SALES_METHOD'])['TOTAL_SALES'].sum()

RETAILER       SALES_METHOD
Amazon         Outlet           9914000.0
Foot Locker    In-store        21320250.0
               Online          21322400.0
               Outlet           8407550.0
Kohl's         In-store         3958900.0
               Online           2289808.0
               Outlet            980425.0
Sports Direct  Online          11966621.0
Walmart        Online           2060394.0
               Outlet            269537.0
West Gear      Online           3588083.0
               Outlet           8990668.0
Name: TOTAL_SALES, dtype: float64

In [116]:
# Calculate sales by month
df.groupby(['YEAR', 'MONTH'])['TOTAL_SALES'].sum()

YEAR  MONTH
2020  1         2037438.0
      2         1999990.0
      3         2153188.0
      4         2763077.0
      5         1963059.0
      6         1011168.0
      7         1962346.0
      8         2323377.0
      9         2058936.0
      10        1306587.0
      11        1149213.0
      12        1051933.0
2021  1         7254659.0
      2         6167647.0
      3         5339798.0
      4         6511142.0
      5         8557112.0
      6         8466356.0
      7         9830212.0
      8         9484586.0
      9         7991722.0
      10        7191664.0
      11        7960322.0
      12       10313416.0
Name: TOTAL_SALES, dtype: float64